Omar Bruce Homework #3 2/26/2026 DATA101

Part I: Import data and describe.

In [27]:
import pandas as pd
df = pd.read_csv('titanic.csv')

print(df.info())
print()
print(df.describe())

variables = df.columns.tolist()
categories = ["categorical nominal","categorical nominal","categorical ordinal","categorical nominal","categorical nominal","numerical","numerical","numerical","categorical nominal","numerical","categorical nominal","categorical nominal"]
print()
print(f"Variable       Category")
print("-" * 35)
for x in range(12):
    print(f"{variables[x]:<15} {categories[x]:>6}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
None

       PassengerId    Survived      Pclass         Age       SibSp  \
count   891.000000  891.000000  891.000000  714.000000  891.000000   
mean    446.000000    0.383838    2.308642   29.699118    0.523008   
std     257.353842    0.4865

Part II: Impute Missing Values

In [3]:
# Impute missing values for Embarked. 
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

#Impute missing values for Age using simple mean.
#df['Age'] = df['Age'].fillna(df['Age'].mean())

#Impute missing values for Age by Sex/Pclass groups

df['Age'] = df['Age'].fillna(df.groupby(['Sex','Pclass'])['Age'].transform('median'))
print(df.loc[:,['Age','Name']])

      Age                                               Name
0    22.0                            Braund, Mr. Owen Harris
1    38.0  Cumings, Mrs. John Bradley (Florence Briggs Th...
2    26.0                             Heikkinen, Miss. Laina
3    35.0       Futrelle, Mrs. Jacques Heath (Lily May Peel)
4    35.0                           Allen, Mr. William Henry
..    ...                                                ...
886  27.0                              Montvila, Rev. Juozas
887  19.0                       Graham, Miss. Margaret Edith
888  21.5           Johnston, Miss. Catherine Helen "Carrie"
889  26.0                              Behr, Mr. Karl Howell
890  32.0                                Dooley, Mr. Patrick

[891 rows x 2 columns]


Part III: Create a FamilySize Variable

In [4]:
#This FamilySize variable will not be accurate. Family members are sometime listed indivually, so there will be overcounting. 

#Create new variable(column) 
df['familysize'] = (df['SibSp'] + df['Parch'] + 1).astype(float)

print(df.head())

print()
print('The smallest family size was:')
print(df['familysize'].min())
print('The largest family size was:')
print(df['familysize'].max())
print('The mean family size was:')
print(f"{df['familysize'].mean():.2f}")
print('The number of people travelling alone was:')
print((df['familysize'] == 1.0).sum())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  familysize  
0      0         A/5 21171   7.2500   NaN        S         2.0  
1      0          PC 17599  71.2833   C85        C         2.0  
2      0  STON/O2. 3101282   7.9250   NaN        S         1.0  
3      0            113803  53.1000  C123        S         2

Part IV: Create a filtered dataframe of all passengers under 16 years old. 

In [30]:
df_youth = df[df['Age'] < 16]

#print(df_youth)

survival_rate = (df_youth['Survived'] == 1).sum() / len(df_youth)
print(f"The Survival Rate for childeren under 16 was:{survival_rate * 100: .2f}%")

df_group = df_youth.groupby(['Sex','Pclass'])['Survived'].mean()

print()
print('Sex           Pclass')
print('-'*30)
print('            1     2     3')
print("female    .667  1.00  .533")
print("male      1.00  1.00  .321") 


The Survival Rate for childeren under 16 was: 59.04%

Sex           Pclass
------------------------------
            1     2     3
female    .667  1.00  .533
male      1.00  1.00  .321


These results tell us a few things. The first is that being in third class lowered your chances of surviving. The second thing we can see is that being a child raised your chances of survival when compared to the adults. And finally, for the children, males and females were treated relatively equally. 

Part V: Compute the average fare for each Pclass. 

In [10]:
#First, remove all of the duplicate ticket numbers, or the mean will be too high. 
deduplicated = df.drop_duplicates(subset=['Ticket'])

pclass_mean = deduplicated.groupby('Pclass')['Fare'].mean()

print(pclass_mean.to_markdown())

|   Pclass |    Fare |
|---------:|--------:|
|        1 | 64.1393 |
|        2 | 17.5097 |
|        3 | 10.0757 |
